# Annotation Confidence Scoring v2

## Goal

For each of the **1,988 named entries** in Oliver's Orbitrap HILIC negESI spreadsheet, score how trustworthy the annotation is.

**Benchmark:** Train on regular TP (936) + FP (215), hold out golden TP (362) + FP (215) as test set. Compare against Oliver's ad hoc probability on the same holdout.

**Production:** Retrain on all labeled data (golden + regular TP + FP), score all 1,988 entries including 475 first-pass annotations Oliver never re-examined.

**Key design choices:**
- All spectral channels use **IK14-matched pairs** — the library hit for the annotation compound specifically, not the top-ranked hit.
- `identity_score_reference_library_` in MassWiki measures the **wrong compound 13.3% of the time** (top ref hit ≠ annotation compound). We use `anno_entropy_sim` instead.
- We filter `hit_source='reference'` to exclude the in-house annotation library (circular evidence).
- Delta RT = observed RT − Kong-predicted RT from annotation SMILES (always the correct compound).

## Data tiers

| Tier | n | Definition |
|---|---|---|
| golden_tp | 362 | Solid TPs ∪ commented ∪ multi-adduct, within rows 0–1297 |
| regular_tp | 936 | Rows 0–1297, reviewed twice, not golden |
| first_pass | 475 | After row 1297, named but never re-examined |
| fp | 215 | `yy_` prefix, confirmed wrong |

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, rankdata
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
import ms_entropy
warnings.filterwarnings('ignore')

# File paths (relative to code/ directory)
SPECTRA_PATH    = '../data/masswiki_Orbitrap HILIC negESI_2026-03-19.xlsx'
HITS_PATH       = '../data/orbitrap_hits_refetched.csv'
SOLID_TP_PATH   = '../data/solid_tp.csv'
INCHIKEY_PATH   = '../data/inchikey_cache.json'
LIB_CACHE_PATH  = '../data/library_peaks_cache.json'
QUERY_CACHE_PATH = '../data/query_peaks_cache.json'
OUTPUT_PATH     = '../results/annotation_confidence_v2'

PPM_TOL = 10.0
# Note: we filter by hit_source='reference' instead of an exclusion list.
# This is cleaner than EXCLUDED_DBS = {'5min_hilic_neg', '5min_lipid_neg'}
# because it automatically excludes all in-house annotation libraries.

print('Imports loaded')

Imports loaded


In [2]:
# ── Load spreadsheet ──
spectra_raw = pd.read_excel(SPECTRA_PATH, header=4)

# Assign base labels: rows 0-1297 are re-annotated TP, yy_ = FP, zz_ = TN
spectra_raw['label'] = 'unlabeled'
spectra_raw.loc[:1297, 'label'] = 'TP'
spectra_raw.loc[spectra_raw['name'].str.startswith('yy_', na=False), 'label'] = 'FP'
spectra_raw.loc[spectra_raw['name'].str.startswith('zz_', na=False), 'label'] = 'TN'

# Remove ISTDs (1_ prefix) and unnamed/TN entries
is_named = (
    spectra_raw['name'].notna() &
    ~spectra_raw['name'].astype(str).str.startswith('zz_') &
    ~spectra_raw['name'].astype(str).str.startswith('1_')
)
spectra = spectra_raw[is_named].copy().reset_index(drop=True)
print(f'Named entries (no zz_, no ISTD): {len(spectra):,}')

# ── Load IK14 cache ──
with open(INCHIKEY_PATH) as f:
    inchikey_cache = json.load(f)

def get_ik14(smiles):
    """Extract IK14 from cache. Values may be dicts or strings."""
    if not smiles or not isinstance(smiles, str):
        return ''
    val = inchikey_cache.get(smiles, '')
    if isinstance(val, dict):
        return val.get('ik14', '')
    return val if isinstance(val, str) else ''

spectra['anno_ik14'] = spectra['smiles'].fillna('').apply(get_ik14)

# ── Define tiers ──
solid_tp_wids = set(pd.read_csv(SOLID_TP_PATH)['wiki_id'])

# Oliver's expert comments (column AK / Unnamed: 36)
comment_col = 'Unnamed: 36'
has_comment = spectra[comment_col].notna() & (spectra[comment_col].astype(str).str.strip() != '')
commented_tp_wids = set(spectra.loc[
    has_comment & (spectra['label'] == 'TP'), 'wiki_id'
])

# Multi-adduct corroboration: same compound name, >=2 distinct adducts, within TP set
tp_spectra = spectra[spectra['label'] == 'TP'].copy()
tp_spectra['name_lower'] = tp_spectra['name'].str.strip().str.lower()
adduct_counts = tp_spectra.groupby('name_lower')['adduct'].nunique()
multi_adduct_names = set(adduct_counts[adduct_counts >= 2].index)
multi_adduct_wids = set(tp_spectra.loc[
    tp_spectra['name_lower'].isin(multi_adduct_names), 'wiki_id'
])

# Golden TP = union of three sources, intersected with TP label
golden_wids = (solid_tp_wids | commented_tp_wids | multi_adduct_wids)
spectra['tier'] = 'first_pass'  # default: named but not re-reviewed
spectra.loc[spectra['label'] == 'TP', 'tier'] = 'regular_tp'
spectra.loc[
    (spectra['label'] == 'TP') & spectra['wiki_id'].isin(golden_wids),
    'tier'
] = 'golden_tp'
spectra.loc[spectra['label'] == 'FP', 'tier'] = 'fp'

print(f'\nTier counts:')
print(f'  golden_tp:  {(spectra["tier"]=="golden_tp").sum():>5d}  (solid: {len(solid_tp_wids)}, commented: {len(commented_tp_wids)}, multi-adduct: {len(multi_adduct_wids)})')
print(f'  regular_tp: {(spectra["tier"]=="regular_tp").sum():>5d}')
print(f'  first_pass: {(spectra["tier"]=="first_pass").sum():>5d}  (unlabeled, never re-examined)')
print(f'  fp:         {(spectra["tier"]=="fp").sum():>5d}')
print(f'  TOTAL:      {len(spectra):>5d}')

Named entries (no zz_, no ISTD): 1,982

Tier counts:
  golden_tp:    385  (solid: 112, commented: 82, multi-adduct: 256)
  regular_tp:   910
  first_pass:   472  (unlabeled, never re-examined)
  fp:           215
  TOTAL:       1982


In [ ]:
# ── Adduct classification using Oliver's taxonomy (2026-04-08) ──
# Source: adduct_list_for_taxonomy_for Ziyue Yang 04-2026.xlsx
# 764 adducts classified as: ok (56), isf (685), dubious (23)
# Note: [M-H]- was overridden from ISF to ok (alphabetical sorting artifact)

adduct_tax = pd.read_csv('../data/adduct_taxonomy_oliver.csv')
adduct_lookup = dict(zip(adduct_tax['adduct'].str.strip(), adduct_tax['category']))

# Map adducts to Oliver's categories
spectra['adduct_cat'] = spectra['adduct'].map(adduct_lookup).fillna('unknown')

# Binary features for the model
spectra['is_isf_adduct'] = (spectra['adduct_cat'] == 'isf').astype(int)
spectra['is_dubious_adduct'] = (spectra['adduct_cat'] == 'dubious').astype(int)

# Per-compound: does this compound have an ok adduct (e.g. [M-H]-) anywhere?
spectra['name_lower'] = spectra['name'].fillna('').str.strip().str.lower()
compound_has_ok = (
    spectra[spectra['adduct_cat'] == 'ok']
    .groupby('name_lower')['wiki_id'].count()
    .rename('_has_ok')
)
spectra = spectra.merge(
    compound_has_ok.reset_index(), on='name_lower', how='left'
)
spectra['has_ok_adduct'] = spectra['_has_ok'].fillna(0).clip(upper=1).astype(int)
spectra.drop(columns='_has_ok', inplace=True)

# ISF-only: spectrum has ISF adduct AND compound lacks ok adduct confirmation
spectra['isf_no_mh'] = ((spectra['is_isf_adduct'] == 1) & 
                         (spectra['has_ok_adduct'] == 0)).astype(int)

# Count distinct adducts per compound
n_adducts = spectra.groupby('name_lower')['adduct'].nunique().rename('n_compound_adducts')
spectra = spectra.merge(n_adducts.reset_index(), on='name_lower', how='left')

# N-acetyl flag (Kong RT systematically wrong for this class -- Oliver 2026-04-08)
spectra['is_nacetyl'] = spectra['name'].fillna('').str.contains(
    r'(?i)^N-?acetyl|^N\d-acetyl|^acetyl-.*(?:amine|alanine|valine|leucine|'
    r'isoleucine|glycine|serine|threonine|cysteine|methionine|phenylalanine|'
    r'tyrosine|tryptophan|aspart|glutam|histid|lysine|arginine|proline|ornithine|'
    r'citrulline|carnosine)'
).astype(int)

# Summary
named = spectra[spectra['label'].isin(['TP', 'FP'])]
print('=== Adduct features (TP+FP, Oliver taxonomy) ===')
print(f'Oliver category:')
print(named['adduct_cat'].value_counts().to_string())
print(f'\nis_isf_adduct:        {named["is_isf_adduct"].sum():>4d} / {len(named)}')
print(f'is_dubious_adduct:    {named["is_dubious_adduct"].sum():>4d} / {len(named)}')
print(f'has_ok_adduct:        {named["has_ok_adduct"].sum():>4d} / {len(named)}')
print(f'isf_no_mh (danger):   {named["isf_no_mh"].sum():>4d} / {len(named)}')
print(f'n_compound_adducts>1: {(named["n_compound_adducts"]>1).sum():>4d} / {len(named)}')
print(f'is_nacetyl:           {named["is_nacetyl"].sum():>4d} / {len(named)}')


In [3]:
# ── Load hits ──
hits_raw = pd.read_csv(HITS_PATH)
print(f'Raw hits: {len(hits_raw):,}')

# Filter to reference hits only (exclude in-house annotation library)
hits = hits_raw[hits_raw['hit_source'] == 'reference'].copy()
print(f'Reference hits: {len(hits):,}')

# Map IK14 for each hit
hits['hit_ik14'] = hits['smiles'].fillna('').apply(get_ik14)

# Join with named spectra
named_info = spectra[['wiki_id', 'anno_ik14', 'label', 'tier']].copy()
joint = hits.merge(named_info, on='wiki_id', how='inner')
print(f'Hits for named spectra: {len(joint):,}')

# Mark annotation hits (IK14 match between hit and annotation)
joint['is_anno_hit'] = (
    (joint['hit_ik14'] != '') &
    (joint['anno_ik14'] != '') &
    (joint['hit_ik14'] == joint['anno_ik14'])
)
print(f'  Annotation hits (IK14 match): {joint["is_anno_hit"].sum():,}')

# Dedup by (wiki_id, hit_ik14) keeping best entropy_similarity
joint['dedup_key'] = joint.apply(
    lambda r: (r['wiki_id'], r['hit_ik14']) if r['hit_ik14'] else (r['wiki_id'], r['lib_name']),
    axis=1
)
before = len(joint)
joint = (
    joint
    .sort_values('entropy_similarity', ascending=False)
    .drop_duplicates(subset='dedup_key')
    .reset_index(drop=True)
)
print(f'\nDeduplication: {before:,} -> {len(joint):,} ({before - len(joint):,} removed)')
print(f'Spectra with annotation hit: {joint[joint["is_anno_hit"]]["wiki_id"].nunique():,}')

Raw hits: 120,828
Reference hits: 107,875
Hits for named spectra: 107,771
  Annotation hits (IK14 match): 36,158

Deduplication: 107,771 -> 13,482 (94,289 removed)
Spectra with annotation hit: 1,482


In [4]:
# ── Load peak caches ──
with open(LIB_CACHE_PATH) as f:
    lib_peaks_cache = json.load(f)
print(f'Library peaks cache: {len(lib_peaks_cache):,} entries')

with open(QUERY_CACHE_PATH) as f:
    query_peaks_cache = json.load(f)
print(f'Query peaks cache: {len(query_peaks_cache):,} entries')

Library peaks cache: 27,015 entries
Query peaks cache: 1,513 entries


In [5]:
def compute_scores(query_peaks_raw, lib_peaks_raw, ppm_tol=PPM_TOL):
    """Compute forward/reverse similarity and peak matching metrics.
    
    Uses ms_entropy preprocessing (clean + weight) to match the internal
    entropy similarity calculation.
    
    Returns dict with: reverse_score, forward_score, n_matched, max_deviation
    or None if computation fails.
    """
    if not query_peaks_raw or not lib_peaks_raw:
        return None
    
    try:
        # Clean and apply entropy weighting
        q_clean = ms_entropy.clean_spectrum(query_peaks_raw)
        l_clean = ms_entropy.clean_spectrum(lib_peaks_raw)
        q_weighted = ms_entropy.apply_weight_to_intensity(q_clean)
        l_weighted = ms_entropy.apply_weight_to_intensity(l_clean)
        
        q_arr = np.array(q_weighted, dtype=float)
        l_arr = np.array(l_weighted, dtype=float)
        if len(q_arr) == 0 or len(l_arr) == 0:
            return None
        
        q_mz, q_int = q_arr[:, 0], q_arr[:, 1]
        l_mz, l_int = l_arr[:, 0], l_arr[:, 1]
        
        # Normalize to sum=1
        q_int_norm = q_int / q_int.sum()
        l_int_norm = l_int / l_int.sum()
        
        # Match peaks within ppm tolerance
        matched_l = 0.0
        matched_q = 0.0
        matched_pairs = []
        used_q = np.zeros(len(q_mz), dtype=bool)
        
        for j, (lm, li) in enumerate(zip(l_mz, l_int_norm)):
            tol = lm * ppm_tol / 1e6
            diffs = np.abs(q_mz - lm)
            candidates = np.where((diffs <= tol) & ~used_q)[0]
            if len(candidates) > 0:
                best = candidates[np.argmin(diffs[candidates])]
                matched_l += li
                matched_q += q_int_norm[best]
                matched_pairs.append((q_int[best], l_int[j]))
                used_q[best] = True
        
        result = {
            'reverse_score': matched_l,   # fraction of library intensity matched
            'forward_score': matched_q,   # fraction of query intensity matched
            'n_matched': len(matched_pairs),
        }
        
        # Max log2 intensity deviation across matched pairs
        if len(matched_pairs) >= 2:
            q_int_arr = np.array([p[0] for p in matched_pairs])
            l_int_arr = np.array([p[1] for p in matched_pairs])
            log_ratios = np.log2((q_int_arr + 1e-6) / (l_int_arr + 1e-6))
            result['max_deviation'] = np.max(np.abs(log_ratios))
        else:
            result['max_deviation'] = np.nan
        
        return result
    except Exception:
        return None

print('compute_scores() defined')

compute_scores() defined


In [11]:
# ── Build evidence table for all named entries ──

# Precompute per-spectrum annotation hit info from joint
# For each wiki_id: best annotation hit (IK14 match), and best non-annotation hit
anno_hit_rows = []
for wid, g in joint.groupby('wiki_id'):
    anno = g[g['is_anno_hit']]
    others = g[~g['is_anno_hit']]
    
    if len(anno) > 0:
        best = anno.loc[anno['entropy_similarity'].idxmax()]
        next_sim = others['entropy_similarity'].max() if len(others) > 0 else 0.0
        anno_rank = (g['entropy_similarity'] >= best['entropy_similarity']).sum()
        
        # Compute forward/reverse for annotation hit
        lib_peaks = lib_peaks_cache.get(best.get('library_wiki_id', ''))
        qry_peaks = query_peaks_cache.get(wid)
        scores = compute_scores(qry_peaks, lib_peaks)
        
        lib_mz = pd.to_numeric(best.get('lib_precursor_mz', np.nan), errors='coerce')
        obs_mz = spectra.loc[spectra['wiki_id']==wid, 'precursor_mz'].iloc[0]
        
        row = {
            'wiki_id': wid,
            'anno_entropy_sim': best['entropy_similarity'],
            'anno_delta_mda': abs(obs_mz - lib_mz) * 1000 if pd.notna(lib_mz) and lib_mz > 0 else np.nan,
            'sim_gap': max(0.0, best['entropy_similarity'] - next_sim) if pd.notna(next_sim) else best['entropy_similarity'],
            'anno_rank': anno_rank,
            'has_anno_hit': True,
        }
        
        if scores:
            row['anno_reverse'] = scores['reverse_score']
            row['anno_forward'] = scores['forward_score']
            row['max_deviation'] = scores['max_deviation']
        else:
            row['anno_reverse'] = np.nan
            row['anno_forward'] = np.nan
            row['max_deviation'] = np.nan
    else:
        row = {
            'wiki_id': wid,
            'anno_entropy_sim': np.nan,
            'anno_delta_mda': np.nan,
            'sim_gap': np.nan,
            'anno_rank': np.nan,
            'anno_reverse': np.nan,
            'anno_forward': np.nan,
            'max_deviation': np.nan,
            'has_anno_hit': False,
        }
    
    anno_hit_rows.append(row)

anno_hit_df = pd.DataFrame(anno_hit_rows)

# ── Check id_score_mismatch ──
top_ref_hits = (
    joint
    .sort_values('entropy_similarity', ascending=False)
    .drop_duplicates(subset='wiki_id')
    [['wiki_id', 'hit_ik14']]
    .rename(columns={'hit_ik14': 'top_hit_ik14'})
)

# ── Build evidence table ──
evidence = spectra[['wiki_id', 'name', 'label', 'tier', 'anno_ik14', 'precursor_mz',
                     'is_isf_adduct', 'is_dubious_adduct', 'has_ok_adduct', 'isf_no_mh',
                     'n_compound_adducts', 'is_nacetyl']].copy()

# Spreadsheet channels
evidence['spectral_entropy'] = pd.to_numeric(spectra['entropy'], errors='coerce')
evidence['delta_rt_abs'] = pd.to_numeric(spectra['anno_delta_rt'], errors='coerce').abs()
evidence['identity_score'] = pd.to_numeric(spectra['identity_score'], errors='coerce')

# Merge annotation hit channels
evidence = evidence.merge(anno_hit_df, on='wiki_id', how='left')

# Merge top hit IK14 for mismatch flag
evidence = evidence.merge(top_ref_hits, on='wiki_id', how='left')
evidence['id_score_mismatch'] = (
    (evidence['anno_ik14'] != '') &
    (evidence['top_hit_ik14'].fillna('') != '') &
    (evidence['anno_ik14'] != evidence['top_hit_ik14'])
)

# ── Print coverage ──
print(f'Evidence table: {len(evidence):,} rows')
print(f'\nChannel coverage:')
channels = ['delta_rt_abs', 'spectral_entropy', 'identity_score',
            'anno_entropy_sim', 'anno_forward', 'anno_reverse',
            'anno_delta_mda', 'sim_gap', 'anno_rank', 'max_deviation',
            'is_isf_adduct', 'isf_no_mh', 'n_compound_adducts', 'is_nacetyl']
for ch in channels:
    n = evidence[ch].notna().sum()
    print(f'  {ch:25s}  {n:>5d}/{len(evidence):,}  ({n/len(evidence):.1%})')

# id_score_mismatch rate
has_both = (evidence['anno_ik14'] != '') & (evidence['top_hit_ik14'].fillna('') != '')
mismatch_rate = evidence.loc[has_both, 'id_score_mismatch'].mean()
print(f'\nid_score_mismatch rate: {mismatch_rate:.1%} of spectra where top ref hit != annotation compound')


Evidence table: 1,982 rows

Channel coverage:
  delta_rt_abs                1940/1,982  (97.9%)
  spectral_entropy            1982/1,982  (100.0%)
  identity_score              1982/1,982  (100.0%)
  anno_entropy_sim            1482/1,982  (74.8%)
  anno_forward                1445/1,982  (72.9%)
  anno_reverse                1445/1,982  (72.9%)
  anno_delta_mda              1482/1,982  (74.8%)
  sim_gap                     1482/1,982  (74.8%)
  anno_rank                   1482/1,982  (74.8%)
  max_deviation               1234/1,982  (62.3%)

id_score_mismatch rate: 9.5% of spectra where top ref hit != annotation compound


In [ ]:
# ── BENCHMARK: Train on regular TP + FP, test on golden holdout ──

FEATURE_COLS = [
    'delta_rt_abs',           # RT deviation (Kong predicted RT)
    'spectral_entropy',       # spectrum complexity
    'anno_entropy_sim',       # MS2 match for annotation compound (IK14-matched)
    'anno_forward',           # reverse score for annotation hit
    'anno_reverse',           # forward score for annotation hit
    'anno_delta_mda',         # mass accuracy of annotation hit (mDa)
    'sim_gap',                # margin over next-best candidate
    'anno_rank',              # rank of annotation hit among all hits
    'max_deviation',          # worst peak intensity deviation
    # Adduct features (Oliver review 2026-04-08)
    'is_isf_adduct',          # spectrum adduct is an in-source fragment
    'isf_no_mh',              # ISF adduct with no ok adduct confirmation (high risk)
    'is_dubious_adduct',      # dubious adduct per Oliver's taxonomy
    'n_compound_adducts',     # number of distinct adducts for this compound
    # RT reliability flag
    'is_nacetyl',             # N-acetyl compound (Kong RT systematically wrong)
]

# ── Config: train on regular TP + FP ──
train_mask = evidence['tier'].isin(['regular_tp', 'fp'])
df_train = evidence[train_mask].copy()
y_train = (df_train['tier'] == 'regular_tp').astype(int)

train_medians = df_train[FEATURE_COLS].median()
X_train = df_train[FEATURE_COLS].fillna(train_medians).values

print(f'Training: {len(df_train)} (regular_tp={y_train.sum()}, FP={(~y_train.astype(bool)).sum()})')
print(f'Features: {len(FEATURE_COLS)}')

# 5-fold CV on training set
gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_probs = cross_val_predict(gb, X_train, y_train, cv=skf, method='predict_proba')[:, 1]
cv_auc = roc_auc_score(y_train, cv_probs)
print(f'5-fold CV AUC: {cv_auc:.3f}')

# Fit on full training set
gb.fit(X_train, y_train)

# ── Holdout: golden TP + FP (never seen during training) ──
test_mask = evidence['tier'].isin(['golden_tp', 'fp'])
df_test = evidence[test_mask].copy()
y_test = (df_test['tier'] == 'golden_tp').astype(int)
X_test = df_test[FEATURE_COLS].fillna(train_medians).values

holdout_probs = gb.predict_proba(X_test)[:, 1]
holdout_auc = roc_auc_score(y_test, holdout_probs)

# Oliver's score on the same holdout set
spectra_oliver = pd.read_excel(SPECTRA_PATH, header=4)[['wiki_id', "oliver's ad hoc probability"]]
spectra_oliver = spectra_oliver.rename(columns={"oliver's ad hoc probability": 'oliver_prob'})
df_test = df_test.merge(spectra_oliver, on='wiki_id', how='left').reset_index(drop=True)
y_test = (df_test['tier'] == 'golden_tp').astype(int)
oliver_valid = df_test['oliver_prob'].notna()
oliver_holdout_auc = roc_auc_score(y_test[oliver_valid], df_test.loc[oliver_valid, 'oliver_prob'])

print(f'\nHeld-out AUC (golden TP vs FP):')
print(f'  Our model:      {holdout_auc:.3f}')
print(f"  Oliver's score:  {oliver_holdout_auc:.3f}")

# ── Feature importance ──
print(f'\nFeature importance:')
for col, imp in sorted(zip(FEATURE_COLS, gb.feature_importances_), key=lambda x: -x[1]):
    print(f'  {col:25s}  {imp:.3f}')

# ── Per-feature AUC ──
print(f'\nPer-feature AUC (univariate, on training set):')
for col in FEATURE_COLS:
    vals = df_train[col].fillna(train_medians[col]).values
    auc_pos = roc_auc_score(y_train, vals)
    auc_neg = roc_auc_score(y_train, -vals)
    best_auc = max(auc_pos, auc_neg)
    direction = 'high=TP' if auc_pos >= auc_neg else 'low=TP'
    print(f'  {col:25s}  AUC={best_auc:.3f}  ({direction})')


Training: 1125 (regular_tp=910, FP=215)
5-fold CV AUC: 0.862

Held-out AUC (golden TP vs FP):
  Our model:      0.930
  Oliver's score:  0.805

Feature importance:
  delta_rt_abs               0.600
  anno_entropy_sim           0.086
  sim_gap                    0.074
  anno_delta_mda             0.068
  max_deviation              0.053
  anno_forward               0.047
  spectral_entropy           0.044
  anno_reverse               0.024
  anno_rank                  0.004

Per-feature AUC (univariate, on training set):
  delta_rt_abs               AUC=0.832  (low=TP)
  spectral_entropy           AUC=0.592  (high=TP)
  anno_entropy_sim           AUC=0.642  (high=TP)
  anno_forward               AUC=0.687  (high=TP)
  anno_reverse               AUC=0.622  (high=TP)
  anno_delta_mda             AUC=0.511  (low=TP)
  sim_gap                    AUC=0.618  (high=TP)
  anno_rank                  AUC=0.550  (low=TP)
  max_deviation              AUC=0.573  (low=TP)


In [9]:
# ── Production scoring: retrain on ALL labeled data, score all 1,988 ──
# For deployment, use all available labeled data (golden + regular TP + FP)

prod_mask = evidence['tier'].isin(['golden_tp', 'regular_tp', 'fp'])
df_prod = evidence[prod_mask].copy()
y_prod = (df_prod['tier'].isin(['golden_tp', 'regular_tp'])).astype(int)

X_prod = df_prod[FEATURE_COLS].fillna(train_medians).values
gb_prod = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
gb_prod.fit(X_prod, y_prod)

print(f'Production model: trained on {len(df_prod)} (TP={y_prod.sum()}, FP={(~y_prod.astype(bool)).sum()})')

# Score ALL 1,988
X_all = evidence[FEATURE_COLS].fillna(train_medians).values
evidence['confidence'] = gb_prod.predict_proba(X_all)[:, 1]

# Score distribution by tier
print(f'\nConfidence score by tier:')
print(f'{"tier":>15s}  {"n":>5s}  {"mean":>6s}  {"median":>6s}  {"P10":>6s}  {"P90":>6s}')
for tier in ['golden_tp', 'regular_tp', 'first_pass', 'fp']:
    sub = evidence[evidence['tier'] == tier]['confidence']
    if len(sub) > 0:
        print(f'{tier:>15s}  {len(sub):>5d}  {sub.mean():>6.3f}  {sub.median():>6.3f}  '
              f'{sub.quantile(0.1):>6.3f}  {sub.quantile(0.9):>6.3f}')

# Flag suspicious
fp_p75 = evidence.loc[evidence['tier'] == 'fp', 'confidence'].quantile(0.75)
print(f'\nFP 75th percentile: {fp_p75:.3f}')

for tier in ['regular_tp', 'first_pass']:
    sub = evidence[evidence['tier'] == tier]
    flagged = sub[sub['confidence'] < fp_p75]
    print(f'{tier}: {len(flagged)}/{len(sub)} ({len(flagged)/len(sub)*100:.1f}%) score below FP P75')
    for _, r in flagged.nsmallest(5, 'confidence').iterrows():
        sim_str = f"sim={r['anno_entropy_sim']:.3f}" if pd.notna(r['anno_entropy_sim']) else "NO LIB HIT"
        rev_str = f"rev={r['anno_forward']:.3f}" if pd.notna(r['anno_forward']) else ""
        print(f'    score={r["confidence"]:.3f}  drt={r["delta_rt_abs"]:.1f}  {sim_str}  {rev_str}  "{str(r["name"])[:30]}"')

Production model: trained on 1510 (TP=1295, FP=215)

Confidence score by tier:
           tier      n    mean  median     P10     P90
      golden_tp    385   0.923   0.960   0.815   0.986
     regular_tp    910   0.939   0.966   0.863   0.987
     first_pass    472   0.596   0.691   0.192   0.914
             fp    215   0.394   0.390   0.045   0.775

FP 75th percentile: 0.587
regular_tp: 9/910 (1.0%) score below FP P75
    score=0.418  drt=26.2  sim=0.694  rev=0.743  "indole-3-acetate-2-sulfonate"
    score=0.513  drt=32.6  sim=0.670  rev=0.458  "nonenedioic acid"
    score=0.521  drt=25.1  sim=0.674  rev=0.391  "octyl sulfate"
    score=0.524  drt=52.0  sim=0.831  rev=0.000  "5-aminonicotinic acid"
    score=0.552  drt=41.7  sim=0.845  rev=0.571  "N-acetylleucine_minor"
first_pass: 207/472 (43.9%) score below FP P75
    score=0.027  drt=67.5  NO LIB HIT    "(2-ethylhexyl)phosphonic acid "
    score=0.027  drt=73.4  NO LIB HIT    "phenol sulfate"
    score=0.031  drt=67.7  NO LIB HIT

In [10]:
# ── Save results ──
os.makedirs(OUTPUT_PATH, exist_ok=True)

output_cols = [
    'wiki_id', 'name', 'label', 'tier', 'confidence',
    # Spreadsheet channels
    'delta_rt_abs', 'spectral_entropy', 'identity_score',
    # IK14-matched annotation hit channels
    'anno_entropy_sim', 'anno_forward', 'anno_reverse',
    'anno_delta_mda', 'sim_gap', 'anno_rank', 'max_deviation',
    # Adduct features
    'is_isf_adduct', 'is_dubious_adduct', 'isf_no_mh', 'n_compound_adducts', 'is_nacetyl',
    # Diagnostic flags
    'has_anno_hit', 'id_score_mismatch',
]

out = evidence[output_cols].sort_values('confidence').copy()
out.to_csv(os.path.join(OUTPUT_PATH, 'annotation_confidence_scores.csv'), index=False)

print(f'Saved {len(out):,} rows -> {OUTPUT_PATH}/annotation_confidence_scores.csv')
print(f'\n=== Summary ===')
print(f'Total named entries:   {len(out):,}')
print(f'Features:              {len(FEATURE_COLS)} (was 9, added adduct + N-acetyl)')
print(f'CV AUC:                {cv_auc:.3f}')
print(f'No annotation hit:     {(evidence["has_anno_hit"] == False).sum()} (spectral pair channels are NaN)')
print(f'id_score_mismatch:     {evidence["id_score_mismatch"].sum()} '
      f'({evidence["id_score_mismatch"].mean():.1%})')


Saved 1,982 rows -> ../results/annotation_confidence_v2/annotation_confidence_scores.csv

=== Summary ===
Total named entries:   1,982
Training set:          600 (golden_tp + fp)
CV AUC:                0.862


TypeError: bad operand type for unary ~: 'float'